# Define Catalog and Schema

In [0]:
%sql
USE CATALOG github_lakehouse;
USE SCHEMA silver;

## Create the `pull_request_review_events` Silver Table

This step creates the Delta table used to store cleaned and structured **PullRequestReviewEvent** records in the Silver layer.

The table contains three groups of fields:

- **Common event fields** — General GitHub event information such as event ID, timestamp, actor, and repository.
- **Bronze lineage fields** — Metadata retained from the Bronze layer to preserve source-level traceability.
- **PullRequestReviewEvent-specific fields** — Information about pull request reviews, including the review action, associated pull request, review ID, and review state.

The `review_state` field can be used to distinguish review outcomes such as approval, requested changes, or comments depending on the values present in the source data.

These fields allow the Silver layer to represent the review activity associated with pull requests in a structured form.

The table is stored using **Delta Lake** (`USING DELTA`).

`CREATE TABLE IF NOT EXISTS` makes the table creation step idempotent and safe to execute multiple times.

> **Note:** This step creates the table structure only. Review-event extraction, normalization, cleaning, and validation are performed in later steps.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.pull_request_review_events (
    -- Common fields
    event_id         STRING,
    event_type       STRING,
    event_timestamp  TIMESTAMP,
    event_date       DATE,
    actor_id         BIGINT,
    actor_login      STRING,
    repo_id          BIGINT,
    repo_name        STRING,

    -- Event-specific fields
    review_action  STRING,
    pr_id          BIGINT,
    pr_number      BIGINT,
    review_id      BIGINT,
    review_state   STRING,

    -- Bronze lineage
    _source_file     STRING,
    _source_date     DATE,
    _ingested_at     TIMESTAMP
)
USING DELTA;